In [24]:
import torch
import pandas as pd
import numpy as np
import regex as re
from epsilon_transformers.persistence import Persister


In [25]:
def analyze_model_ranks(model):
    results=[]
    for name, param in model.named_parameters():
        tensor=param.detach()
        if tensor.dim()<2:
            continue
        if tensor.dim()==2:
            rank=torch.linalg.matrix_rank(tensor).item()
            full_rank=min(tensor.shape)
            results.append({
                "parameter": name,
                "shape": tuple(tensor.shape),
                "rank": rank,
                "full_rank": full_rank,
                "deficiency": full_rank - rank
            })
        elif tensor.dim()==4:
            n_heads=tensor.shape[0]
            for head_idx in range(n_heads):
                head_mat=tensor[head_idx]
                rank=torch.linalg.matrix_rank(head_mat).item()
                full_rank=min(head_mat.shape)
                results.append({
                    "parameter": f"{name}_head_{head_idx}",
                    "shape": tuple(head_mat.shape),
                    "rank": rank,
                    "full_rank": full_rank,
                    "deficiency": full_rank - rank
                })
    return pd.DataFrame(results)                

In [26]:
persister = Persister(save_dir="/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/small checkpoints/1lyr_0.15_0.6_lr0.01adm_40M_400/epsilon-transformers/models/linrmess3_thry/1lyr_0.15_0.6_lr0.01adm_40M_400/")
model_path="/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/small checkpoints/1lyr_0.15_0.6_lr0.01adm_40M_400/epsilon-transformers/models/linrmess3_thry/1lyr_0.15_0.6_lr0.01adm_40M_400/checkpoint_801_tokens_40000000.pt"
model = persister.load_model(model_path)

In [27]:
from epsilon_transformers.process.processes import PROCESS_REGISTRY
from torch import device
process_name = 'Linear_Mess3'
process_params ={
    "x": 0.15,
    "a": 0.6
}
seq_len = 10
vocab = 3
if process_name in PROCESS_REGISTRY:
    process=PROCESS_REGISTRY[process_name](**process_params)
history=process.generate_process_history(total_length=10)
if torch.cuda.is_available():
    device = device("cuda:0")
elif torch.backends.mps.is_available():
    device = device("mps")
else:
    device = device("cpu")
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)    

In [28]:
df_norms = measure_residual_norms(model, input_seq)

NameError: name 'measure_residual_norms' is not defined

In [ ]:
print(df_norms)

   layer_idx                     hook_name        component       norm
0          0       blocks.0.hook_resid_pre   hook_resid_pre   6.600586
1          0  blocks.0.ln1.hook_normalized  hook_normalized   1.183700
2          0        blocks.0.hook_attn_out    hook_attn_out   2.031928
3          0       blocks.0.hook_resid_mid   hook_resid_mid   7.960988
4          0  blocks.0.ln2.hook_normalized  hook_normalized   2.234404
5          0         blocks.0.mlp.hook_pre         hook_pre   5.544907
6          0        blocks.0.mlp.hook_post        hook_post   0.546068
7          0         blocks.0.hook_mlp_out     hook_mlp_out   4.530251
8          0      blocks.0.hook_resid_post  hook_resid_post  10.491097
9         -1      ln_final.hook_normalized  hook_normalized   0.432001


In [ ]:
print(df_ranks)
print("\nLow Rank Matrices:\n", df_ranks[df_ranks["deficiency"] > 0])

            parameter    shape  rank  full_rank  deficiency
0           embed.W_E   (3, 3)     3          3           0
1     pos_embed.W_pos  (10, 3)     3          3           0
2   blocks.0.attn.b_Q   (1, 3)     1          1           0
3   blocks.0.attn.b_K   (1, 3)     1          1           0
4   blocks.0.attn.b_V   (1, 3)     1          1           0
5   blocks.0.mlp.W_in  (3, 12)     3          3           0
6  blocks.0.mlp.W_out  (12, 3)     3          3           0
7         unembed.W_U   (3, 3)     3          3           0

Low Rank Matrices:
 Empty DataFrame
Columns: [parameter, shape, rank, full_rank, deficiency]
Index: []


In [ ]:
print(df_ranks)
print("\nLow Rank Matrices:\n", df_ranks[df_ranks["deficiency"] > 0])

             parameter      shape  rank  full_rank  deficiency
0            embed.W_E    (3, 64)     3          3           0
1      pos_embed.W_pos   (10, 64)    10         10           0
2    blocks.0.attn.b_Q     (2, 8)     2          2           0
3    blocks.0.attn.b_K     (2, 8)     2          2           0
4    blocks.0.attn.b_V     (2, 8)     2          2           0
5    blocks.0.mlp.W_in  (64, 256)    64         64           0
6   blocks.0.mlp.W_out  (256, 64)    64         64           0
7    blocks.1.attn.b_Q     (2, 8)     2          2           0
8    blocks.1.attn.b_K     (2, 8)     2          2           0
9    blocks.1.attn.b_V     (2, 8)     2          2           0
10   blocks.1.mlp.W_in  (64, 256)    64         64           0
11  blocks.1.mlp.W_out  (256, 64)    64         64           0
12         unembed.W_U    (64, 3)     3          3           0

Low Rank Matrices:
 Empty DataFrame
Columns: [parameter, shape, rank, full_rank, deficiency]
Index: []


In [ ]:
print(df_ranks)
print("\nLow Rank Matrices:\n", df_ranks[df_ranks["deficiency"] > 0])

             parameter      shape  rank  full_rank  deficiency
0            embed.W_E    (3, 64)     3          3           0
1      pos_embed.W_pos   (10, 64)    10         10           0
2    blocks.0.attn.b_Q     (2, 8)     2          2           0
3    blocks.0.attn.b_K     (2, 8)     2          2           0
4    blocks.0.attn.b_V     (2, 8)     2          2           0
5    blocks.0.mlp.W_in  (64, 256)    64         64           0
6   blocks.0.mlp.W_out  (256, 64)    64         64           0
7    blocks.1.attn.b_Q     (2, 8)     2          2           0
8    blocks.1.attn.b_K     (2, 8)     2          2           0
9    blocks.1.attn.b_V     (2, 8)     2          2           0
10   blocks.1.mlp.W_in  (64, 256)    64         64           0
11  blocks.1.mlp.W_out  (256, 64)    64         64           0
12         unembed.W_U    (64, 3)     3          3           0

Low Rank Matrices:
 Empty DataFrame
Columns: [parameter, shape, rank, full_rank, deficiency]
Index: []


In [ ]:
print(df_ranks)
print("\nLow Rank Matrices:\n", df_ranks[df_ranks["deficiency"] > 0])

             parameter      shape  rank  full_rank  deficiency
0            embed.W_E    (3, 64)     3          3           0
1      pos_embed.W_pos   (10, 64)    10         10           0
2    blocks.0.attn.b_Q     (2, 8)     2          2           0
3    blocks.0.attn.b_K     (2, 8)     2          2           0
4    blocks.0.attn.b_V     (2, 8)     2          2           0
5    blocks.0.mlp.W_in  (64, 256)    64         64           0
6   blocks.0.mlp.W_out  (256, 64)    64         64           0
7    blocks.1.attn.b_Q     (2, 8)     2          2           0
8    blocks.1.attn.b_K     (2, 8)     2          2           0
9    blocks.1.attn.b_V     (2, 8)     2          2           0
10   blocks.1.mlp.W_in  (64, 256)    64         64           0
11  blocks.1.mlp.W_out  (256, 64)    64         64           0
12         unembed.W_U    (64, 3)     3          3           0

Low Rank Matrices:
 Empty DataFrame
Columns: [parameter, shape, rank, full_rank, deficiency]
Index: []


In [ ]:
def measure_residual_norms(model, input_seq,layer_idx=None):
    def filter(name):
        return any(name.endswith(suffix) for suffix in ['out','_normalized','_pre','_post','_mid'])
    with torch.no_grad():
        _,cache=model.run_with_cache(input_seq,names_filter=filter)
    norms=[]
    for hook_name, act in cache.items():
        if act.dim()>=2:
            norm_val=act.flatten(start_dim=2).norm(dim=-1).mean().item()
        else:
            norm_val=act.norm().item()
        match=re.search(r'blocks\.(\d+)\.', hook_name)
        layer_idx=int(match.group(1)) if match else -1
        component=hook_name.split('.')[-1]
        norms.append({
            "layer_idx": layer_idx,
            "hook_name": hook_name,
            "component": component,
            "norm": norm_val  })
    return pd.DataFrame(norms)

In [ ]:
def ablation(model, input_seq,hook_name):
    def zero_hook(activations,hook):
        return torch.zeros_like(activations)
    with model.hooks(fwd_hooks=[(hook_name,zero_hook)]):
        logits=model(input_seq)
    return logits

In [ ]:
input_seq=torch.tensor([[1,1,0,2,1,1,1,2,2,2]],dtype=torch.long)

In [ ]:
with torch.no_grad():
    _, cache=model.run_with_cache(input_seq)
hook_name='blocks.0.hook_mlp_out'    
cache_act=cache[hook_name]
print(cache_act)
norm=cache_act.norm().item()
print(f"Norm of activations at {hook_name}: {norm}")
variance=cache_act.var().item()
print(f"Variance of activations at {hook_name}: {variance}")

tensor([[[-2.7619,  0.4948, -0.1775,  2.5151, -1.5817,  1.9888,  0.1695,
           1.6366, -1.9397,  0.0229,  3.1662, -0.6829, -1.8779,  1.3264,
          -1.2645,  0.2821,  1.8683, -0.6908,  1.4961,  0.1264, -1.0220,
           2.4129,  2.7328, -0.1900,  0.9680, -0.0261,  0.2288, -0.0255,
           0.3880,  0.2383, -1.2034, -0.2064, -0.9974, -1.4505,  0.3415,
          -2.1154, -0.6625, -0.9574, -0.1475, -0.4544,  0.9122,  0.9663,
          -0.2540,  0.6001,  2.9935,  0.8185,  0.7452,  1.8387,  1.0216,
           1.2007,  2.2094, -0.2765, -0.4427, -0.1249, -0.5228, -1.1856,
          -0.3545,  1.5152, -0.3307, -0.4498,  0.8130, -0.7449, -1.0524,
          -0.2044]]])
Norm of activations at blocks.0.hook_mlp_out: 10.361372947692871
Variance of activations at blocks.0.hook_mlp_out: 1.6703804731369019


In [ ]:
with torch.no_grad():
    _, cache=model.run_with_cache(input_seq)
hook_name='blocks.0.hook_mlp_out'    
cache_act=cache[hook_name]
print(cache_act)
norm=cache_act.norm().item()
print(f"Norm of activations at {hook_name}: {norm}")
variance=cache_act.var().item()
print(f"Variance of activations at {hook_name}: {variance}")

tensor([[[-1.3950, -1.4883,  2.5038],
         [-1.7147, -2.9241,  7.0138],
         [-1.3950, -1.4883,  2.5038],
         [-0.6574, -6.1377,  5.8825],
         [-1.3950, -1.4883,  2.5038],
         [-1.3950, -1.4883,  2.5038],
         [-1.3950, -1.4883,  2.5038],
         [-0.6096, -7.0399,  7.1422],
         [-0.6165, -6.9003,  6.9438],
         [-0.6255, -6.7216,  6.6918]]])
Norm of activations at blocks.0.hook_mlp_out: 21.749683380126953
Variance of activations at blocks.0.hook_mlp_out: 16.306612014770508


tensor([[[-1.3615,  1.5457, -0.8380,  0.2007,  0.4802, -0.0870, -1.6374,
           2.1768,  0.6627,  1.5227,  0.7640, -0.4300, -1.3383,  0.1751,
          -0.8854, -1.7988,  0.6675, -0.2757,  1.1209, -0.6538, -0.5206,
          -0.0750,  0.9160, -0.8097, -0.2248, -1.5375,  0.6916,  0.9069,
           0.6678,  0.3281,  0.6525,  0.5110, -2.0519,  1.5188, -1.1874,
          -2.4656, -0.6901,  0.0657, -1.0311,  1.0094,  0.9483,  0.3592,
          -0.0220,  1.0619, -2.5305,  2.0560,  1.0988,  1.7447, -1.7507,
           2.5790, -2.1695, -1.6673, -1.0942,  0.1840, -0.1283, -1.5977,
           0.2466, -1.0082, -1.1547, -0.8117, -1.5243, -1.2212,  3.0809,
          -1.2129]]])
Norm of activations at blocks.1.hook_mlp_out: 10.168623924255371
Variance of activations at blocks.1.hook_mlp_out: 1.6260031461715698


In [ ]:
with torch.no_grad():
    _, cache=model.run_with_cache(input_seq)
hook_name_1='blocks.1.hook_mlp_out'    
cache_act_1=cache[hook_name_1]
hook_name_0='blocks.0.hook_mlp_out'
cache_act=cache[hook_name_0]
diff=cache_act_1 - cache_act
print(cache_act)
norm=diff.norm().item()
print(f"Norm of difference in activations between {hook_name_1} and {hook_name_0}: {norm}")
variance=diff.var().item()
print(f"Variance of difference in activations between {hook_name_1} and {hook_name_0}: {variance}")

tensor([[[-2.7619,  0.4948, -0.1775,  2.5151, -1.5817,  1.9888,  0.1695,
           1.6366, -1.9397,  0.0229,  3.1662, -0.6829, -1.8779,  1.3264,
          -1.2645,  0.2821,  1.8683, -0.6908,  1.4961,  0.1264, -1.0220,
           2.4129,  2.7328, -0.1900,  0.9680, -0.0261,  0.2288, -0.0255,
           0.3880,  0.2383, -1.2034, -0.2064, -0.9974, -1.4505,  0.3415,
          -2.1154, -0.6625, -0.9574, -0.1475, -0.4544,  0.9122,  0.9663,
          -0.2540,  0.6001,  2.9935,  0.8185,  0.7452,  1.8387,  1.0216,
           1.2007,  2.2094, -0.2765, -0.4427, -0.1249, -0.5228, -1.1856,
          -0.3545,  1.5152, -0.3307, -0.4498,  0.8130, -0.7449, -1.0524,
          -0.2044]]])
Norm of difference in activations between blocks.1.hook_mlp_out and blocks.0.hook_mlp_out: 13.239885330200195
Variance of difference in activations between blocks.1.hook_mlp_out and blocks.0.hook_mlp_out: 2.6880593299865723


In [ ]:
df_norms = measure_residual_norms(model, input_seq)
print(df_norms)
#chekpt0

    layer_idx                     hook_name        component       norm
0           0       blocks.0.hook_resid_pre   hook_resid_pre   1.297573
1           0  blocks.0.ln1.hook_normalized  hook_normalized   7.990040
2           0        blocks.0.hook_attn_out    hook_attn_out   3.065934
3           0       blocks.0.hook_resid_mid   hook_resid_mid   3.448415
4           0  blocks.0.ln2.hook_normalized  hook_normalized   8.036119
5           0         blocks.0.mlp.hook_pre         hook_pre  14.750746
6           0        blocks.0.mlp.hook_post        hook_post  11.006336
7           0         blocks.0.hook_mlp_out     hook_mlp_out  10.361373
8           0      blocks.0.hook_resid_post  hook_resid_post  10.660256
9           1       blocks.1.hook_resid_pre   hook_resid_pre  10.660256
10          1  blocks.1.ln1.hook_normalized  hook_normalized   7.998356
11          1        blocks.1.hook_attn_out    hook_attn_out   2.170738
12          1       blocks.1.hook_resid_mid   hook_resid_mid  11

In [ ]:
df_norms = measure_residual_norms(model, input_seq)
print(df_norms)
#chekptfinal

    layer_idx                     hook_name        component       norm
0           0       blocks.0.hook_resid_pre   hook_resid_pre   1.117845
1           0  blocks.0.ln1.hook_normalized  hook_normalized   8.000730
2           0        blocks.0.hook_attn_out    hook_attn_out   1.764405
3           0       blocks.0.hook_resid_mid   hook_resid_mid   2.067586
4           0  blocks.0.ln2.hook_normalized  hook_normalized   8.008543
5           0         blocks.0.mlp.hook_pre         hook_pre  12.904223
6           0        blocks.0.mlp.hook_post        hook_post   9.253723
7           0         blocks.0.hook_mlp_out     hook_mlp_out   7.866004
8           0      blocks.0.hook_resid_post  hook_resid_post   8.257267
9           1       blocks.1.hook_resid_pre   hook_resid_pre   8.257267
10          1  blocks.1.ln1.hook_normalized  hook_normalized   8.000715
11          1        blocks.1.hook_attn_out    hook_attn_out   1.840300
12          1       blocks.1.hook_resid_mid   hook_resid_mid   8

In [ ]:
df_norms = measure_residual_norms(model, input_seq)
print(df_norms)
#linear model

    layer_idx                     hook_name        component       norm
0           0       blocks.0.hook_resid_pre   hook_resid_pre   1.198434
1           0  blocks.0.ln1.hook_normalized  hook_normalized   7.994504
2           0        blocks.0.hook_attn_out    hook_attn_out   2.269022
3           0       blocks.0.hook_resid_mid   hook_resid_mid   2.591191
4           0  blocks.0.ln2.hook_normalized  hook_normalized   8.032362
5           0         blocks.0.mlp.hook_pre         hook_pre  13.921023
6           0        blocks.0.mlp.hook_post        hook_post  10.077938
7           0         blocks.0.hook_mlp_out     hook_mlp_out   8.696688
8           0      blocks.0.hook_resid_post  hook_resid_post   9.229602
9           1       blocks.1.hook_resid_pre   hook_resid_pre   9.229602
10          1  blocks.1.ln1.hook_normalized  hook_normalized   7.998013
11          1        blocks.1.hook_attn_out    hook_attn_out   1.919553
12          1       blocks.1.hook_resid_mid   hook_resid_mid   9

In [ ]:
import torch
import numpy as np
from sklearn.linear_model import LinearRegression

def analyze_mlp_math(model, process, layer_idx=0, num_seqs=128, seq_len=10, device=None):
    """
    Check whether the MLP at layer `layer_idx` implements a token‑dependent
    linear map:  mlp_out ≈ M_z * mlp_in for each token z in {0,1,2}.
    """
    if device is None:
        if torch.backends.mps.is_available():
            device = torch.device("mps")
        else:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Generate a batch of sequences: [B, T]
    if hasattr(process, "generate_batch_gpu"):
        input_batch = process.generate_batch_gpu(
            batch_size=num_seqs, seq_len=seq_len, device=device
        )
    else:
        seqs = [
            process.generate_process_history(total_length=seq_len).symbols
            for _ in range(num_seqs)
        ]
        input_batch = torch.tensor(seqs, dtype=torch.long, device=device)

    # 2. Run model, cache LN2 (MLP input) and MLP out for that layer
    cache_names = [
        f"blocks.{layer_idx}.ln2.hook_normalized",
        f"blocks.{layer_idx}.hook_mlp_out",
    ]
    with torch.no_grad():
        _, cache = model.run_with_cache(input_batch, names_filter=cache_names)

    # Shapes: [B, T, d_model]
    mlp_in = cache[cache_names[0]].cpu().numpy()
    mlp_out = cache[cache_names[1]].cpu().numpy()
    tokens = input_batch.cpu().numpy()  # [B, T]

    # Flatten batch + time: [B*T, ...]
    B, T, D = mlp_in.shape
    mlp_in_flat = mlp_in.reshape(B * T, D)
    mlp_out_flat = mlp_out.reshape(B * T, D)
    tokens_flat = tokens.reshape(B * T)

    results = {}

    for token_id in range(3):  # tokens 0,1,2 for Mess3
        mask = (tokens_flat == token_id)
        X = mlp_in_flat[mask]
        Y = mlp_out_flat[mask]

        if X.shape[0] < D:  # need enough samples
            print(f"Token {token_id}: not enough samples ({X.shape[0]}), skipping.")
            continue

        # Fit linear map Y ≈ X @ M  (no bias; LN should remove mean shifts)
        reg = LinearRegression(fit_intercept=False).fit(X, Y)
        M_learned = reg.coef_        # [D, D]
        r2 = reg.score(X, Y)

        results[token_id] = {"M": M_learned, "R2": r2}
        print(f"Token {token_id}: MLP linear fit R^2 = {r2:.4f}")

    return results

# Example call:
# res = analyze_mlp_math(model, process, layer_idx=1, num_seqs=256, seq_len=32)


In [ ]:
res = analyze_mlp_math(model, process, layer_idx=0, num_seqs=256, seq_len=10)

Token 0: MLP linear fit R^2 = 0.7344
Token 1: MLP linear fit R^2 = 0.7207
Token 2: MLP linear fit R^2 = 0.9982


In [ ]:
# in the end we want to do linear regression between the activations and the transformer_input_beliefs
def run_activation_to_beliefs_regression(activations, ground_truth_beliefs):

    # make sure the first two dimensions are the same
    assert activations.shape[0] == ground_truth_beliefs.shape[0]
    assert activations.shape[1] == ground_truth_beliefs.shape[1]

    # flatten the activations
    batch_size, n_ctx, d_model = activations.shape
    belief_dim = ground_truth_beliefs.shape[-1]
    activations_flattened = activations.view(-1, d_model) # [batch * n_ctx, d_model]
    ground_truth_beliefs_flattened = ground_truth_beliefs.view(-1, belief_dim) # [batch * n_ctx, belief_dim]
    
    # run the regression
    regression = LinearRegression()
    regression.fit(activations_flattened, ground_truth_beliefs_flattened)

    # get the belief predictions
    belief_predictions = regression.predict(activations_flattened) # [batch * n_ctx, belief_dim]
    belief_predictions = belief_predictions.reshape(batch_size, n_ctx, belief_dim)

    return regression, belief_predictions



In [32]:
def get_mlp_io_and_beliefs(model, process, regression, layer_idx=0, num_seqs=256, seq_len=10):
    """
    Generates NEW random data, grabs MLP inputs/outputs, and uses the 
    PRE-TRAINED regression probe to estimate beliefs for them.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Generate NEW random sequences (broad distribution)
    if hasattr(process, "generate_batch_gpu"):
        inputs = process.generate_batch_gpu(batch_size=num_seqs, seq_len=seq_len, device=device)
    else:
        seqs = [process.generate_process_history(total_length=seq_len).symbols for _ in range(num_seqs)]
        inputs = torch.tensor(seqs, dtype=torch.long, device=device)

    # 2. Run Model
    # We need resid_mid (or whatever layer you trained the probe on) to project to beliefs
    # And MLP input/output for analysis.
    # Note: If you trained probe on Layer 0 resid_post, you should use that. 
    # If analyzing Layer 1 MLP, we usually project Layer 1 resid_mid to beliefs.
    # Let's assume the probe is valid for the layer we are analyzing (or the space is shared).
    cache_names = [
        f"blocks.{layer_idx}.ln2.hook_normalized", # MLP In
        f"blocks.{layer_idx}.hook_mlp_out",         # MLP Out
        f"blocks.{layer_idx}.hook_resid_mid",       # For Belief Projection
    ]
    with torch.no_grad():
        _, cache = model.run_with_cache(inputs, names_filter=cache_names)

    mlp_in = cache[cache_names[0]].cpu().numpy()
    mlp_out = cache[cache_names[1]].cpu().numpy()
    resid_for_probe = cache[cache_names[2]].cpu().numpy()
    tokens = inputs.cpu().numpy()

    # Flatten
    B, T, D = mlp_in.shape
    mlp_in_flat = mlp_in.reshape(B * T, D)
    mlp_out_flat = mlp_out.reshape(B * T, D)
    resid_flat = resid_for_probe.reshape(B * T, D)
    tokens_flat = tokens.reshape(B * T)

    # 3. PROJECT Beliefs using your pre-trained probe
    # regression.predict expects [N, d_model]
    beliefs_flat = regression.predict(resid_flat) # [N, 3]

    return mlp_in_flat, mlp_out_flat, tokens_flat, beliefs_flat


In [35]:
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
import numpy as np

# --- 1. Per-Region Linearity ---
def analyze_per_region_linearity(model, process, regression, layer_idx=1, num_seqs=256, seq_len=32, n_clusters=3):
    print("\n--- 1. Per-Region Linearity Analysis ---")
    mlp_in, mlp_out, tokens, _ = get_mlp_io_and_beliefs(model, process, regression, layer_idx=0, num_seqs=256, seq_len=10)
    
    for token_id in range(3):
        mask = (tokens == token_id)
        X = mlp_in[mask]
        Y = mlp_out[mask]
        
        if len(X) < 50: continue
            
        print(f"Token {token_id}: Global R^2 = {LinearRegression(fit_intercept=False).fit(X, Y).score(X, Y):.4f}")
        
        # Cluster
        kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
        labels = kmeans.fit_predict(X)
        
        for i in range(n_clusters):
            c_mask = (labels == i)
            if c_mask.sum() < 20: continue
            r2 = LinearRegression(fit_intercept=True).fit(X[c_mask], Y[c_mask]).score(X[c_mask], Y[c_mask])
            print(f"  Cluster {i} (n={c_mask.sum()}): R^2 = {r2:.4f}")

# --- 2. Error vs Entropy ---
def analyze_error_entropy(model, process, regression, layer_idx=1, num_seqs=256, seq_len=32):
    print("\n--- 2. Error vs Belief Entropy ---")
    mlp_in, mlp_out, tokens, beliefs = get_mlp_io_and_beliefs(model, process, regression, layer_idx, num_seqs, seq_len)
    
    results = {}
    for token_id in range(3):
        mask = (tokens == token_id)
        X, Y, B = mlp_in[mask], mlp_out[mask], beliefs[mask]
        if len(X) < 50: continue
            
        reg = LinearRegression(fit_intercept=False).fit(X, Y)
        residuals = Y - reg.predict(X)
        e_norm = np.linalg.norm(residuals, axis=1)
        
        # Entropy
        B_safe = np.clip(B, 1e-9, 1.0)
        # Normalize rows to sum to 1 just in case probe output drifted
        B_safe = B_safe / B_safe.sum(axis=1, keepdims=True)
        entropy = -np.sum(B_safe * np.log(B_safe), axis=1)
        
        corr = np.corrcoef(e_norm, entropy)[0, 1]
        print(f"Token {token_id}: Corr(Error, Entropy) = {corr:.4f}")
        results[token_id] = {"reg": reg} # Save for next step

    return results

# --- 3. Orthogonality ---
def analyze_orthogonality(error_results, regression):
    print("\n--- 3. Error Orthogonality ---")
    # P: Belief subspace basis [3, d_model] (from your probe)
    P = regression.coef_ 
    Q, _ = np.linalg.qr(P.T) # Orthonormal basis [d_model, 3]
    
    # We need residuals again. For simplicity, let's re-extract valid data or assume passed.
    # To keep it self-contained, I will assume we are analyzing the *general property* using generic data
    # But usually you pass the residuals from Step 2.
    # Here is a cleaner way:
    # Just inspect the ratio for a fresh batch.
    mlp_in, mlp_out, tokens, _ = get_mlp_io_and_beliefs(model, process, regression)
    
    for token_id in range(3):
        mask = (tokens == token_id)
        if mask.sum() < 50: continue
            
        # Re-fit/Predict to get 'e'
        reg = LinearRegression(fit_intercept=False).fit(mlp_in[mask], mlp_out[mask])
        e = mlp_out[mask] - reg.predict(mlp_in[mask])
        
        # Project e onto belief subspace Q
        e_in_belief = e @ Q @ Q.T
        e_ortho = e - e_in_belief
        
        norm_in = np.linalg.norm(e_in_belief, axis=1).mean()
        norm_ortho = np.linalg.norm(e_ortho, axis=1).mean()
        ratio = norm_in / (norm_in + norm_ortho)
        
        print(f"Token {token_id}: Ratio (Error in Belief Space / Total Error) = {ratio:.4f}")

# --- 4. Matrix Comparison ---
def compare_matrices(model, process, regression, layer_idx=1):
    print("\n--- 4. Learned vs Theoretical Matrix ---")
    P = regression.coef_ # [3, d]
    P_pinv = np.linalg.pinv(P) # [d, 3]
    
    # Get Theoretical S Matrices from Process
    T_theory, _ = process._create_hmm() # [3, 3, 3] (Token, From, To) usually
    
    mlp_in, mlp_out, tokens, _ = get_mlp_io_and_beliefs(model, process, regression, layer_idx)
    
    for token_id in range(3):
        mask = (tokens == token_id)
        if mask.sum() < 50: continue
            
        reg = LinearRegression(fit_intercept=False).fit(mlp_in[mask], mlp_out[mask])
        M_z = reg.coef_ # [d, d]
        
        # Project: P M P+
        M_eff = P @ M_z @ P_pinv
        
        # Theoretical S^z
        # Check process definition. Usually T[token] is the transition matrix.
        # But attention/belief update uses Transpose or Row-stochastic form.
        # Mess3 paper: r_new = r_old * S^z. 
        # If your vectors are row vectors (x @ M), then S^z is T[token].
        # If column vectors (M @ x), it's T[token].T.
        # Standard LinearRegression fits y = x @ A.T + b. So coef_ is A (output x input).
        # Your probe P maps x -> belief.
        # So M_eff maps belief -> belief_update.
        
        S_z = T_theory[token_id] # Adjust transpose if needed based on convention
        
        print(f"\nToken {token_id}:")
        print("Learned (3x3):\n", np.round(M_eff, 3))
        print("Theory (S^z):\n", np.round(S_z, 3))


In [ ]:
from epsilon_transformers.analysis.activation_analysis import get_beliefs_for_transformer_inputs
mixed_state_tree = process.derive_mixed_state_presentation(depth=10 + 1)
MSP_transition_matrix = mixed_state_tree.build_msp_transition_matrix()

# in order to plot the belief states in the simplex, we need to get the paths and beliefs from the MSP
tree_paths, tree_beliefs = mixed_state_tree.paths_and_belief_states
msp_beliefs = [tuple(round(b, 5) for b in belief) for belief in tree_beliefs]
print(f"Number of Unique beliefs: {len(set(msp_beliefs))} out of {len(msp_beliefs)}")

# now lets index each belief
msp_belief_index = {b: i for i, b in enumerate(set(msp_beliefs))}
device = 'cpu'
train_config = persister.load_training_config()
transformer_inputs = [x for x in tree_paths if len(x) == 10]
transformer_inputs = torch.tensor(transformer_inputs, dtype=torch.int).to(device)

# print first few batches
print(transformer_inputs[:5])


transformer_input_beliefs, transformer_input_belief_indices = get_beliefs_for_transformer_inputs(transformer_inputs, msp_belief_index, tree_paths, tree_beliefs)
print(f"Transformer Input Beliefs: {transformer_input_beliefs.shape}, Transformer Input Belief Indices: {transformer_input_belief_indices.shape}")

_, activations = model.run_with_cache(transformer_inputs, names_filter=lambda x: 'resid_mid' in x)
#_, activations = model.run_with_cache(transformer_inputs)
#activations['blocks.0.hook_resid_mid'].shape  'ln_final.hook_normalized'
#activations = activations['blocks.3.hook_resid_post']
activations.keys()
print(activations.keys())
#acts = torch.concatenate((activations["blocks.0.ln1.hook_normalized"], activations["blocks.1.ln1.hook_normalized"], activations["blocks.2.ln1.hook_normalized"], activations["blocks.3.ln1.hook_normalized"]), dim=-1)
#acts = activations['ln_final.hook_normalized']
acts = activations['blocks.0.hook_resid_mid']
regression, belief_predictions = run_activation_to_beliefs_regression(acts, transformer_input_beliefs)
print(belief_predictions.shape)

Number of Unique beliefs: 265720 out of 265720
tensor([[2, 0, 2, 2, 0, 0, 0, 0, 0, 0],
        [2, 0, 2, 2, 0, 0, 0, 0, 0, 1],
        [2, 0, 2, 2, 0, 0, 0, 0, 0, 2],
        [1, 1, 1, 1, 1, 1, 2, 2, 2, 0],
        [1, 1, 1, 1, 1, 1, 2, 2, 2, 1]], dtype=torch.int32)
Transformer Input Beliefs: torch.Size([59049, 10, 3]), Transformer Input Belief Indices: torch.Size([59049, 10])
dict_keys(['blocks.0.hook_resid_mid'])


NameError: name 'run_activation_to_beliefs_regression' is not defined

In [23]:
regression, belief_predictions = run_activation_to_beliefs_regression(acts, transformer_input_beliefs)
print(belief_predictions.shape)

(59049, 10, 3)


In [36]:
analyze_per_region_linearity(model, process, regression)


--- 1. Per-Region Linearity Analysis ---
Token 0: Global R^2 = 0.7411
  Cluster 0 (n=147): R^2 = 0.0000
  Cluster 1 (n=81): R^2 = 1.0000
  Cluster 2 (n=585): R^2 = 0.0000
Token 1: Global R^2 = 0.7267
  Cluster 0 (n=83): R^2 = 1.0000
  Cluster 1 (n=446): R^2 = 0.0000
  Cluster 2 (n=364): R^2 = 0.0000
Token 2: Global R^2 = 0.9982
  Cluster 0 (n=672): R^2 = 1.0000
  Cluster 1 (n=93): R^2 = 1.0000
  Cluster 2 (n=89): R^2 = 0.9920


In [37]:
def inspect_variance(model, process, regression, layer_idx=1, num_seqs=256, seq_len=32, n_clusters=3):
    print("\n--- Variance Inspection ---")
    mlp_in, mlp_out, tokens, _ = get_mlp_io_and_beliefs(model, process, regression, layer_idx, num_seqs, seq_len)
    
    for token_id in range(3):
        mask = (tokens == token_id)
        X = mlp_in[mask]
        Y = mlp_out[mask]
        if len(X) < 50: continue

        kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
        labels = kmeans.fit_predict(X)
        
        print(f"\nToken {token_id}:")
        for i in range(n_clusters):
            c_mask = (labels == i)
            Y_c = Y[c_mask]
            if len(Y_c) < 10: continue
            
            # Std Dev of Output Norms
            y_std = Y_c.std(axis=0).mean()
            y_norm = np.linalg.norm(Y_c, axis=1).mean()
            
            # Linearity check again
            reg = LinearRegression(fit_intercept=True).fit(X[c_mask], Y_c)
            r2 = reg.score(X[c_mask], Y_c)
            
            print(f"  Cluster {i}: R2={r2:.4f} | Y_std={y_std:.4f} | Y_norm={y_norm:.4f}")
inspect_variance(model, process, regression, layer_idx=0, num_seqs=256, seq_len=10)


--- Variance Inspection ---

Token 0:
  Cluster 0: R2=0.0000 | Y_std=0.0000 | Y_norm=3.2296
  Cluster 1: R2=1.0000 | Y_std=2.1105 | Y_norm=13.6967
  Cluster 2: R2=0.0000 | Y_std=0.0000 | Y_norm=3.2296

Token 1:
  Cluster 0: R2=0.0000 | Y_std=0.0000 | Y_norm=3.2296
  Cluster 1: R2=1.0000 | Y_std=0.0314 | Y_norm=7.8435
  Cluster 2: R2=0.0000 | Y_std=0.0000 | Y_norm=3.2296

Token 2:
  Cluster 0: R2=1.0000 | Y_std=0.4769 | Y_norm=8.8709
  Cluster 1: R2=0.9948 | Y_std=0.0000 | Y_norm=4.7147
  Cluster 2: R2=1.0000 | Y_std=0.4772 | Y_norm=15.9516


In [38]:
inspect_variance(model, process, regression, layer_idx=0, num_seqs=256, seq_len=10,n_clusters=5)


--- Variance Inspection ---

Token 0:
  Cluster 0: R2=0.0000 | Y_std=0.0000 | Y_norm=3.2296
  Cluster 1: R2=1.0000 | Y_std=0.5894 | Y_norm=11.3602
  Cluster 2: R2=0.0000 | Y_std=0.0000 | Y_norm=3.2296
  Cluster 3: R2=0.0000 | Y_std=0.0000 | Y_norm=3.2296
  Cluster 4: R2=1.0000 | Y_std=0.0000 | Y_norm=18.5346

Token 1:
  Cluster 0: R2=0.0000 | Y_std=0.0000 | Y_norm=3.2296
  Cluster 1: R2=1.0000 | Y_std=0.0302 | Y_norm=7.8368
  Cluster 2: R2=0.0000 | Y_std=0.0000 | Y_norm=3.2296
  Cluster 3: R2=0.0000 | Y_std=0.0000 | Y_norm=3.2296
  Cluster 4: R2=0.0000 | Y_std=0.0000 | Y_norm=3.2296

Token 2:
  Cluster 0: R2=1.0000 | Y_std=0.2737 | Y_norm=10.2319
  Cluster 1: R2=1.0000 | Y_std=0.3979 | Y_norm=16.1433
  Cluster 2: R2=1.0000 | Y_std=0.0000 | Y_norm=4.7147
  Cluster 3: R2=1.0000 | Y_std=0.1978 | Y_norm=7.6185
  Cluster 4: R2=1.0000 | Y_std=0.1704 | Y_norm=9.0241


In [43]:
def get_null_vector(model, process, regression, layer_idx=0):
    """
    Computes the canonical 'Null Vector' by averaging outputs from the Null Cluster.
    """
    # Generate a batch to find the null cluster
    mlp_in, mlp_out, tokens, _ = get_mlp_io_and_beliefs(model, process, regression, layer_idx, num_seqs=128)
    
    # We saw Token 0 has a null cluster. Let's find it.
    mask = (tokens == 0)
    X = mlp_in[mask]
    Y = mlp_out[mask]
    
    # Cluster to separate Linear vs Null
    from sklearn.cluster import KMeans
    kmeans = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
    
    # Identify Null Cluster: The one with lowest variance (Y_std approx 0)
    best_std = float('inf')
    null_center = None
    
    for i in range(3):
        c_mask = (kmeans.labels_ == i)
        if c_mask.sum() < 10: continue
        
        Y_c = Y[c_mask]
        y_std = Y_c.std(axis=0).mean()
        
        if y_std < 0.01: # Threshold for "Constant"
            null_center = Y_c.mean(axis=0) # This is vector 'c'
            best_std = y_std
            
    if null_center is None:
        raise ValueError("Could not find a Null cluster with 0 variance!")
        
    return null_center

def analyze_sequence_robust(model, process, regression, input_seq, null_vector, layer_idx=0):
    # Ensure input is [1, seq_len]
    if input_seq.dim() == 1:
        input_seq = input_seq.unsqueeze(0)

    cache_names = [f"blocks.{layer_idx}.hook_mlp_out", f"blocks.{layer_idx}.hook_resid_mid"]
    with torch.no_grad():
        _, cache = model.run_with_cache(input_seq, names_filter=cache_names)

    mlp_out = cache[cache_names[0]][0].cpu().numpy() # [seq_len, d_model]
    resid_mid = cache[cache_names[1]][0].cpu().numpy()
    
    # FIX 1: Extract the single sequence properly
    tokens = input_seq[0].cpu().numpy() # [seq_len]
    
    pred_beliefs = regression.predict(resid_mid)
    
    # Setup Theory
    T_theory, _ = process._create_hmm()
    current_belief = np.array([1/3, 1/3, 1/3]) # Start uniform
    
    print(f"\n{'Pos':<4} {'Tok':<4} {'Dist to Null':<12} {'Mode':<10} {'Pred Belief':<25}")
    print("-" * 80)

    for i, token in enumerate(tokens):
        # FIX 2: Ensure token is scalar integer for indexing
        token_val = int(token)
        
        # Update Theory
        S_z = T_theory[token_val]
        # Normalize previous belief before update? Usually belief is normalized.
        # Linear update: r' = r S.
        current_belief = current_belief @ S_z 
        # Normalize to keep it a valid probability distribution for comparison
        if current_belief.sum() > 0:
            current_belief /= current_belief.sum()
        
        # Robust Mode Check
        out_vec = mlp_out[i]
        dist = np.linalg.norm(out_vec - null_vector)
        dist_val = float(dist)
        
        # If distance is tiny, it's the Null Vector
        if dist_val < 0.5: # Relaxed threshold slightly (was 0.1)
            mode = "NULL"
        else:
            mode = "LINEAR"
            
        p_str = f"[{pred_beliefs[i][0]:.2f}, {pred_beliefs[i][1]:.2f}, {pred_beliefs[i][2]:.2f}]"
        
        print(f"{i:<4} {token_val:<4} {dist_val:<12.4f} {mode:<10} {p_str:<25}")

# No changes needed to get_null_vector, it looked correct.

# Usage:
history=process.generate_process_history(total_length=10)
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)
c = get_null_vector(model, process, regression)
analyze_sequence_robust(model, process, regression, input_seq=input_seq, null_vector=c)



Pos  Tok  Dist to Null Mode       Pred Belief              
--------------------------------------------------------------------------------
0    0    0.0000       NULL       [0.49, 0.26, 0.25]       
1    0    9.5106       LINEAR     [0.23, 0.41, 0.35]       
2    1    0.0000       NULL       [0.35, 0.46, 0.19]       
3    2    6.7732       LINEAR     [0.28, 0.27, 0.45]       
4    1    0.0000       NULL       [0.33, 0.47, 0.20]       
5    0    0.0000       NULL       [0.39, 0.32, 0.29]       
6    2    7.1384       LINEAR     [0.27, 0.25, 0.48]       
7    1    0.0000       NULL       [0.32, 0.46, 0.22]       
8    1    0.0000       NULL       [0.31, 0.48, 0.21]       
9    2    7.4757       LINEAR     [0.27, 0.24, 0.49]       


In [46]:
def analyze_sequence_with_theory_corrected(model, process, regression, input_seq, null_vector, layer_idx=0):
    if input_seq.dim() == 1: input_seq = input_seq.unsqueeze(0)

    # 1. Run Model
    cache_names = [f"blocks.{layer_idx}.hook_mlp_out", f"blocks.{layer_idx}.hook_resid_mid"]
    with torch.no_grad():
        _, cache = model.run_with_cache(input_seq, names_filter=cache_names)

    mlp_out = cache[cache_names[0]][0].cpu().numpy()
    resid_mid = cache[cache_names[1]][0].cpu().numpy()
    tokens = input_seq[0].cpu().numpy()
    
    pred_beliefs = regression.predict(resid_mid)
    
    # 2. Setup Theory Matrices
    # Raw T for decay sum (sum(T_raw) is stochastic)
    T_raw, _ = process._create_hmm() 
    T_decay = T_raw.sum(axis=0) # This should be the stochastic matrix T
    
    # Normalized S for Linear Update (row stochastic individually)
    # process.create_norm_matrix might not be exposed as public, check if it works
    # If not, implement logic here or use the private method if accessible
    if hasattr(process, "_create_norm_matrix"):
        S_norm = process._create_norm_matrix()
    else:
        # Fallback: Normalize T_raw manually
        S_norm = np.zeros_like(T_raw)
        for z in range(3):
            row_sums = T_raw[z].sum(axis=1, keepdims=True)
            S_norm[z] = T_raw[z] / row_sums

    # Initial Stationary Dist (pi)
    pi = np.array([1/3, 1/3, 1/3])
    
    curr_lin_belief = pi.copy()
    
    print(f"\n{'Pos':<3} {'Tok':<3} {'Mode':<6} {'DiffNorm':<8} {'Pred(Con)':<20} {'Theory(Lin)':<20} {'Theory(Con)':<20}")
    print("-" * 105)

    for d, token in enumerate(tokens):
        token_val = int(token)
        pos = d + 1 
        
        # A. Update Linear Belief (Target): pi * S^z
        # Use S_norm here
        S_z = S_norm[token_val]
        curr_lin_belief = curr_lin_belief @ S_z
        
        # B. Calculate Constrained Belief (Input to MLP)
        # Formula: pi + Sum_{i=1}^d (pi * S^{z_i} * T^{d-i} - pi)
        # Note: Paper says S^zi in the sum.
        
        curr_con_belief = pi.copy()
        
        for i in range(pos): 
            z_i = int(tokens[i])
            S_zi = S_norm[z_i] # Use S_norm for the step
            
            # Decay term T^{d-i} uses the stochastic sum matrix
            power = (d - i) # d is current index (0..9). i is step index (0..d).
            # If i=d (current step), power=0.
            
            T_pow = np.linalg.matrix_power(T_decay, power)
            
            term = (pi @ S_zi @ T_pow) - pi
            curr_con_belief += term
            
        # C. Analyze MLP Mode
        out_vec = mlp_out[d]
        dist = float(np.linalg.norm(out_vec - null_vector))
        mode = "NULL" if dist < 0.5 else "LIN"
        
        # D. Ideal Diff
        theory_diff = curr_lin_belief - curr_con_belief
        theory_diff_norm = np.linalg.norm(theory_diff)
        
        def fmt(v): return f"[{v[0]:.2f} {v[1]:.2f} {v[2]:.2f}]"
        
        print(f"{d:<3} {token_val:<3} {dist:<6.2f} {mode:<5} {theory_diff_norm:<10.4f} {fmt(pred_beliefs[d]):<20} {fmt(curr_lin_belief):<20} {fmt(curr_con_belief):<20}")


In [47]:
# Usage:
history=process.generate_process_history(total_length=10)
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)
c = get_null_vector(model, process, regression)

analyze_sequence_with_theory_corrected(model, process, regression, input_seq=input_seq, null_vector=c)


Pos Tok Mode   DiffNorm Pred(Con)            Theory(Lin)          Theory(Con)         
---------------------------------------------------------------------------------------------------------
0   1   0.00   NULL  0.0000     [0.39 0.48 0.13]     [0.24 0.52 0.24]     [0.24 0.52 0.24]    
1   0   6.81   LIN   0.0256     [0.20 0.48 0.33]     [0.47 0.32 0.20]     [0.47 0.34 0.19]    
2   0   0.00   NULL  0.0249     [0.40 0.33 0.26]     [0.60 0.23 0.18]     [0.60 0.24 0.16]    
3   0   0.00   NULL  0.0199     [0.41 0.31 0.28]     [0.66 0.18 0.16]     [0.67 0.19 0.14]    
4   1   0.00   NULL  0.0540     [0.35 0.45 0.21]     [0.39 0.44 0.17]     [0.42 0.44 0.13]    
5   2   8.23   LIN   0.0317     [0.26 0.27 0.47]     [0.27 0.29 0.44]     [0.29 0.30 0.41]    
6   0   0.00   NULL  0.0109     [0.38 0.33 0.29]     [0.49 0.23 0.29]     [0.50 0.22 0.28]    
7   0   0.00   NULL  0.0116     [0.39 0.31 0.31]     [0.60 0.18 0.21]     [0.61 0.18 0.21]    
8   0   0.00   NULL  0.0137     [0.39 0.30 0.3

In [15]:
res = analyze_mlp_math(model, process)

tensor([[2, 0, 1, 1, 0, 1, 0, 1, 2, 2]], device='mps:0')
nothing
nothing
nothing


In [11]:
print(res)

{}
